<font color=#009afa size=50>DAT-6002 - Fundamentals of Artificial Intelligence</font><br>
<font size=50>WRIT1 Part 2 - Neural Networks</font><br>
<font size=20 color=#00c382>From Scratch MLP - Bank Data</font><br>

|**Assessment Code:** |WRIT1|
|---|---|
|**Author:**|Tom Rowan|
|**Course:**|DAT6002_S2_25|
|**Course Tutor:**|Imtiaz Hussain Khan, PhD|


**Coding Syntax Note: _variable_name --> temporary / locally used variable, discarded after use.**

https://archive.ics.uci.edu/dataset/222/bank+marketing

---

# Initialise Notebook

In [184]:
# Install Requirements Using Pip
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


### Import Python Libraries

In [185]:
# Standard Libraries
import sys
import time
import json
import random
from datetime import datetime

# Pandas and numpy
import pandas as pd
import numpy as np

# Matplot Lib
import matplotlib.pyplot as plt
from matplotlib import colormaps
import matplotlib.colors as mcolors
from matplotlib.pyplot import figure
import matplotlib.ticker as mticker
import matplotlib.animation as animation
%matplotlib inline

# Seaborn Graphs
import seaborn as sns

# Plotly
import plotly.express as px
import plotly.graph_objects as go

# Tabulate
from tabulate import tabulate

# Colorama
from colorama import Fore, Back, Style

# Requests
import requests

# Pycountry 
import pycountry

# GeoPy
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# Cartopy Maps
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Geopandas
import geopandas as gpd

# IPython Display 
from IPython.display import HTML

# Do not print warnings!
import warnings
warnings.filterwarnings('ignore')

### Create a Timestamp

In [186]:
# Generate Timestamp
today = pd.Timestamp(datetime.today())
now_date_timestamp = str(today.strftime("%Y%m%d"))
now_datetime_timestamp = str(today.strftime("%Y%m%d-%H%M"))

### Functions for Pretty Output

print_output - Print coloured Output

In [187]:
# Function to print coloured output
def print_output (str1, str2, delim=':', colour='yellow'):
    if colour.upper() in dir(Fore):
        colour = colour.upper()
    else:
        colour = 'YELLOW'
    print (f'{getattr(Fore, colour)}{str(str1)}{delim}{Style.RESET_ALL} {str(str2)}')

### Check for Internet Access

In [188]:
_urls_to_check = ['http://connectivitycheck.gstatic.com/generate_204',
                  'https://www.github.com',
                  'https://datascience.daemonchild.com']

Set this to False if you have a local dataset

In [189]:
internet_required = False

In [190]:
# Function to Safely Check for Internet Connectivity
def check_internet (url_list):
    online = False
    for _url in url_list:
        try:
            response = requests.get(_url, timeout=3)
            if response.status_code == 200:
                online = True
                break
        except requests.RequestException:
            output_string = f"Check Internet failed for {_url}"
            print_output ('[FAIL]',output_string, colour='red', delim='')
    return online

# Perform the checks
if check_internet (_urls_to_check ):
    print_output ('[INTERNET]','Internet Connectivity Detected. Using Internet resources.',colour='green', delim='')
    online = True

else:
    print ()
    print_output ('[INTERNET]','No Internet Connectivity.', colour='red', delim='')
    online = False


# Stop running if Internet is required
if online == False & internet_required == True:
    sys.exit('Stopping notebook run here.')    


[INTERNET] Internet Connectivity Detected. Using Internet resources.


---
# Load and Explore the Dataset

From: https://archive.ics.uci.edu/dataset/222/bank+marketing

The data is related with direct marketing campaigns of a Portuguese banking institution. The marketing campaigns were based on phone calls. Often, more than one contact to the same client was required, in order to access if the product (bank term deposit) would be ('yes') or not ('no') subscribed. 

[This dataset is] bank-full.csv with all examples and 17 inputs, ordered by date (older version of this dataset with less inputs). 

The classification goal is to predict if the client will subscribe (yes/no) a term deposit (variable y).

|Variable Name	|Role	|Type	|Demographic|	Description|	Units|	Missing Values|
|---|---|---|---|---|---|---|
|age|	Feature|	Integer|	Age|	|	|	no|
|job|	Feature	|Categorical	|Occupation	|type of job (categorical: 'admin.','blue-collar','entrepreneur','housemaid','management','retired','self-employed','services','student','technician','unemployed','unknown')||		no|
|marital	|Feature|	Categorical|	Marital Status|	marital status (categorical: 'divorced','married','single','unknown'; note: 'divorced' means divorced or widowed)||		no|
|education	|Feature|	Categorical	|Education Level|	(categorical: 'basic.4y','basic.6y','basic.9y','high.school','illiterate','professional.course','university.degree','unknown')||		no|
|default	|Feature|	Binary		||has credit in default?||		no|
|balance	|Feature|	Integer		||average yearly balance|	euros	|no|
|housing	|Feature|	Binary		||has housing loan?||		no|
|loan	|Feature|	Binary	||	has personal loan?||		no|
|contact	|Feature|	Categorical	||	contact communication type (categorical: 'cellular','telephone')||		yes|
|day	|Feature|	Integer	||	last contact day of the month||		unknown|
|month	|Feature|	Date	||	last contact month of the year||		unknown|
|duration	|Feature|	Integer	||	last contact duration, in seconds (numeric)||		unknown|
|campaign	|Feature|	Integer	||	number of contacts performed during this campaign and for this client (numeric, includes last contact)||		unknown|
|pdays	|Feature|	Integer	||number of days that passed by after the client was last contacted from a previous campaign (numeric, -1 means client was not previously contacted)||		unknown|
|previous	|Feature|	Integer	||number of contacts performed before this campaign and for this client (numeric)||		unknown|
|poutcome	|Feature|	Categorical	|| outcome of the previous marketing campaign (categorical: "unknown","other","failure","success")||		unknown|



In [191]:
if online:
    # Fetch the dataset from the Internet
    df = pd.read_csv('bank-full.csv', delimiter=';')
else:
    # Load the dataset from local file
    df = pd.read_csv('bank-full.csv', delimiter=';')

### Explore the dataset

In [192]:
print (df.columns)
print ("")
print (df.shape)

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'y'],
      dtype='str')

(45211, 17)


In [193]:
df.sample(10)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
33408,51,blue-collar,married,primary,no,1016,yes,no,cellular,20,apr,78,1,-1,0,unknown,no
32163,58,retired,divorced,primary,no,2745,no,yes,cellular,15,apr,469,1,-1,0,unknown,no
24109,43,retired,married,secondary,no,860,no,no,telephone,30,oct,132,1,-1,0,unknown,yes
38645,41,management,married,tertiary,no,30,yes,no,cellular,15,may,101,4,291,6,failure,no
696,40,admin.,married,secondary,no,6,yes,no,unknown,6,may,60,3,-1,0,unknown,no
18005,49,entrepreneur,divorced,unknown,yes,-701,yes,no,cellular,30,jul,988,2,-1,0,unknown,yes
32732,49,management,married,tertiary,no,5795,yes,no,cellular,17,apr,75,1,-1,0,unknown,no
25986,37,management,single,tertiary,no,5340,yes,no,cellular,19,nov,101,2,104,4,failure,no
23331,44,management,married,tertiary,no,1138,no,no,cellular,27,aug,523,3,-1,0,unknown,yes
38232,35,unemployed,married,tertiary,no,1849,yes,no,cellular,15,may,111,1,-1,0,unknown,no


In [194]:
df.dropna(inplace=True)
df.shape

(45211, 17)

# Conversion and Encoding Ready for the MLP

In [195]:
numerical_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
categorical_cols = ['job', 'marital', 'education', 'contact', 'poutcome']

# Binary columns that should not be one-hot encoded
binary_columns = ['default', 'housing', 'loan', 'y']

# Columns to exclude from the system
excluded_features = ['duration','campaign','pdays','previous']
df.drop(columns=excluded_features, inplace=True)

In [196]:
# Encode the Binary Features
for col in binary_columns:
    # Convert yes/no outcome to float versions
    df[col] = df[col].map({'yes': 1.0, 'no': 0.0}).astype(float)

# One Hot Encode the categorical features
df_one_hot = pd.get_dummies(df)

In [197]:
df_one_hot.sample(10)

,age,default,balance,housing,loan,day,y,job_admin.,job_blue-collar,job_entrepreneur,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_failure,poutcome_other,poutcome_success,poutcome_unknown
34897,42,0.0,-380,1.0,0.0,6,0.0,False,False,False,...,False,False,True,False,False,False,False,True,False,False
34445,31,1.0,10,1.0,1.0,5,0.0,False,False,False,...,False,False,True,False,False,False,False,False,False,True
8775,25,0.0,556,1.0,1.0,4,0.0,False,False,False,...,True,False,False,False,False,False,False,False,False,True
6744,28,0.0,916,1.0,0.0,28,0.0,False,True,False,...,False,False,True,False,False,False,False,False,False,True
40883,37,0.0,3420,0.0,0.0,12,0.0,True,False,False,...,False,False,False,False,False,False,False,False,False,True
30422,33,0.0,28,0.0,1.0,5,0.0,False,True,False,...,False,False,False,False,False,False,False,False,False,True
16645,53,1.0,-252,0.0,1.0,24,0.0,False,False,False,...,False,False,False,False,False,False,False,False,False,True
4292,37,0.0,1633,1.0,0.0,19,0.0,False,False,False,...,False,False,True,False,False,False,False,False,False,True
43804,58,0.0,139,0.0,0.0,27,0.0,False,False,False,...,False,False,True,False,False,False,False,True,False,False
1650,35,0.0,503,1.0,1.0,9,0.0,False,True,False,...,False,False,True,False,False,False,False,False,False,True


In [198]:
# Reencode the True/False values as floats (True = 1.0, False = 0.0)
df_one_hot = df_one_hot.astype(float)
df_one_hot.sample(10)

,age,default,balance,housing,loan,day,y,job_admin.,job_blue-collar,job_entrepreneur,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_failure,poutcome_other,poutcome_success,poutcome_unknown
9799,50.0,0.0,1595.0,0.0,0.0,9.0,1.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
22991,31.0,0.0,218.0,0.0,0.0,26.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
39596,52.0,0.0,195.0,1.0,0.0,26.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
16012,32.0,0.0,0.0,1.0,0.0,22.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
24070,45.0,0.0,1144.0,1.0,0.0,22.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
22710,43.0,0.0,246.0,0.0,0.0,25.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
8931,54.0,0.0,2957.0,1.0,0.0,4.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
23384,53.0,0.0,28.0,0.0,0.0,27.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
24867,59.0,0.0,2429.0,0.0,1.0,18.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
16038,55.0,0.0,255.0,0.0,0.0,22.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


### Define Normalisation and Label Encoding Functions

The goal is that all features contribute roughly equally to the training of the neural network.

$ x_scaled = \frac{x - min(x)}{max(x)-min(x)}$



In [199]:
def max_min_normalize (df, column_name):
    max_value = df[column_name].max()
    min_value = df[column_name].min()
    df[column_name] = (df[column_name] - min_value) / (max_value - min_value)
    return df

# Scale each column, one at a time
for col in df_one_hot.columns.to_list():
    df_scaled = max_min_normalize(df_one_hot, col)


In [200]:
df_scaled.sample(10)

,age,default,balance,housing,loan,day,y,job_admin.,job_blue-collar,job_entrepreneur,...,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_failure,poutcome_other,poutcome_success,poutcome_unknown
42124,0.337662,0.0,0.089309,1.0,1.0,0.066667,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
11311,0.545455,0.0,0.076571,0.0,0.0,0.566667,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
42616,0.610390,0.0,0.075527,0.0,0.0,0.366667,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
38013,0.480519,0.0,0.070815,1.0,0.0,0.433333,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
39412,0.103896,0.0,0.078432,0.0,0.0,0.700000,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
32992,0.181818,0.0,0.075509,1.0,0.0,0.533333,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
40422,0.155844,0.0,0.081247,0.0,0.0,0.033333,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
28366,0.207792,0.0,0.076262,1.0,0.0,0.933333,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
25242,0.220779,0.0,0.073911,1.0,0.0,0.566667,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
35252,0.233766,0.0,0.072803,1.0,1.0,0.200000,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


Write out these arrays.

In [201]:
# Create filenames, one timestamped, one 'latest'
filepath = "."

filename = f'{filepath}/bank_full_scaled_{now_date_timestamp}.csv'
latest_filename = f'{filepath}/bank_full_scaled_latest.csv'

# Write out CSV files
df_scaled.to_csv(filename, index=False)
df_scaled.to_csv(latest_filename, index=False)


---